# arXiv HTTP 406 investigation: live experiments

This notebook replays, with live calls against the real arXiv API, the experiments behind [`docs/406.md`](../../docs/406.md) (issues [#37](https://github.com/pkuppens/production-agentic-rag-course/issues/37), [#39](https://github.com/pkuppens/production-agentic-rag-course/issues/39), [#44](https://github.com/pkuppens/production-agentic-rag-course/issues/44), [#46](https://github.com/pkuppens/production-agentic-rag-course/issues/46)).

`docs/406.md` explains *what* was found. This notebook shows *how* it was found, so a reader can reproduce every hypothesis test themselves against the live API, instead of trusting a write-up alone.

**Be considerate of arXiv's real, shared server.** Every cell below makes a small, bounded number of live requests. Do not increase the trial counts casually, and do not run this notebook in a tight loop.

## Setup

In [ ]:
import asyncio
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path

import httpx

current_dir = Path.cwd()
if current_dir.name == "week2" and current_dir.parent.name == "notebooks":
    project_root = current_dir.parent.parent
elif (current_dir / "compose.yml").exists():
    project_root = current_dir
else:
    raise RuntimeError("Could not find project root")
sys.path.insert(0, str(project_root))

from src.config import ArxivSettings  # noqa: E402
from src.services.arxiv.client import ArxivClient  # noqa: E402

YESTERDAY = (datetime.now() - timedelta(days=1)).strftime("%Y%m%d")
print(f"project_root={project_root}")
print(f"YESTERDAY={YESTERDAY}")

## 1. Direct probe: what does arXiv's own cache header say?

Before testing retries, look at the actual response headers arXiv sends back. `x-cache` tells us whether arXiv served the request from its shared cache (`HIT`) or had to handle it itself (`MISS`). This bypasses `ArxivClient`'s retry logic entirely — three plain HTTP calls, so it stays cheap.

In [ ]:
async def probe(label: str, url: str) -> None:
    async with httpx.AsyncClient(timeout=30) as client:
        start = time.time()
        try:
            response = await client.get(url)
            elapsed = time.time() - start
            xcache = response.headers.get("x-cache", "(none)")
            print(f"{label:34s} status={response.status_code} x-cache={xcache!r:20s} elapsed={elapsed:.1f}s")
        except Exception as e:
            elapsed = time.time() - start
            print(f"{label:34s} FAILED after {elapsed:.1f}s: {type(e).__name__}: {e}")

BASE = "https://export.arxiv.org/api/query"

await probe(
    "broad, no date filter",
    f"{BASE}?search_query=cat:cs.AI&max_results=5&sortBy=submittedDate&sortOrder=descending",
)
await probe(
    "arXiv doc's own example query",
    f"{BASE}?search_query=all:electron+AND+submittedDate:%5B202301010600+TO+202401010600%5D",
)
await probe(
    "production shape: single day",
    f"{BASE}?search_query=cat:cs.AI+AND+submittedDate:%5B{YESTERDAY}0000+TO+{YESTERDAY}2359%5D&max_results=5",
)

**What to expect:** the broad query comes back `x-cache: MISS, HIT` (or just succeeds) because it's popular enough that someone else's identical request already warmed arXiv's cache. Both `submittedDate:[...]` queries come back `x-cache: MISS, MISS` with a `406` — including arXiv's own documented example query, verbatim from their user manual. That rules out "our query is malformed": a uniquely-parameterized date range almost never hits arXiv's cache, and a cache-miss gets a bare 406 while arXiv is throttling this host.

## 2. Hypothesis: is the retry budget too short?

Issue #39 saw a real overload window: 8/8 attempts failed with 406 over ~70s, then the very next attempt succeeded. That raised the question: does `ArxivClient` just give up too early? The cell below replays the exact production query shape (`cat:cs.AI AND submittedDate:[<one day>]`, via `fetch_papers_with_query` so we can reach the original, since-fixed query shape on purpose) under two different retry configurations, using the real `ArxivClient` retry/backoff logic unmodified.

**This is slow by design** — each failing trial exhausts its full retry budget before giving up, the same as it did in production. Expect this cell to take several minutes.

In [ ]:
PRODUCTION_SHAPE_QUERY = f"cat:cs.AI AND submittedDate:[{YESTERDAY}0000 TO {YESTERDAY}2359]"


async def replay_retry_config(label: str, settings: ArxivSettings, trials: int = 1) -> None:
    client = ArxivClient(settings)
    for i in range(1, trials + 1):
        start = time.time()
        try:
            papers = await client.fetch_papers_with_query(PRODUCTION_SHAPE_QUERY, max_results=5)
            elapsed = time.time() - start
            print(f"[{label} #{i}] OK in {elapsed:.1f}s - {len(papers)} papers")
        except Exception as e:
            elapsed = time.time() - start
            print(f"[{label} #{i}] FAILED after {elapsed:.1f}s: {type(e).__name__}")


# Config A: fast, fixed-ish interval - the backoff formula is
# min(rate_limit_delay * 2**attempt, metadata_max_retry_delay), so a low
# retry_delay cap makes it flatten out near that cap quickly (~20s steps).
config_a = ArxivSettings(rate_limit_delay=20.0, metadata_max_retries=4, metadata_max_retry_delay=20.0)

# Config B: slower, wider ceiling - closer to the 10s/30s/.../90s shape
# raised as a hypothesis (exact multiplier is fixed at 2x in the client,
# so this widens the cap and attempt count rather than the step size).
config_b = ArxivSettings(rate_limit_delay=10.0, metadata_max_retries=6, metadata_max_retry_delay=90.0)

await replay_retry_config("config_a (~20s steps, 4 attempts)", config_a)
await replay_retry_config("config_b (up to 90s steps, 6 attempts)", config_b)

### What this looked like at full scale

The cell above uses one trial per config to keep this notebook cheap to run. The investigation behind #46 ran the same production-shape query **6 times**, with an even more generous ceiling (20 attempts, ~10s cap), to see whether *any* realistic retry budget would ever recover:

| Arm | Result |
| --- | --- |
| wide (broad, no date filter) | **6 / 6 succeeded**, ~3s each, attempt 1 |
| chunked (single day, production shape) | **0 / 6 succeeded** — every trial used all 19 of 20 allowed attempts and still failed, ~185s each |

Max attempts needed across all 6 chunked trials: 19 (out of a 20-attempt ceiling — i.e. even a ceiling *more than double* the production default of 8 was not enough).

### Conclusion: rejected

Retrying longer does not fix this. 0/6 trials recovered even with a 20-attempt budget (2.5x the production default) and ~185s of waiting each. More attempts just means a longer wait before the same failure — the query shape itself is the problem, not the retry budget.

## 3. New hypothesis: is it about *small requests*, not date ranges?

A different theory: maybe arXiv has trouble with **small result sets** — a short date range, or very few results — rather than the date-range syntax itself. This section isolates "small/narrow" from "has a `submittedDate:[...]` clause" by testing them independently.

Four shapes, one call each:

1. **broad, `max_results=25`** — control: no date range, many results.
2. **broad, `max_results=1`** — small result count, still *no* date range.
3. **wide date range** (a full year) — has a `submittedDate:[...]` clause, but is not a *single day*.
4. **single-day date range** — the production shape: both narrow *and* has a date-range clause.

In [ ]:
await probe("broad, max_results=25 (control)",
            f"{BASE}?search_query=cat:cs.AI&max_results=25&sortBy=submittedDate&sortOrder=descending")
await probe("broad, max_results=1 (small, no date)",
            f"{BASE}?search_query=cat:cs.AI&max_results=1&sortBy=submittedDate&sortOrder=descending")
await probe("wide date range (1 year)",
            f"{BASE}?search_query=cat:cs.AI+AND+submittedDate:%5B20250101+TO+20260101%5D&max_results=25")
await probe("single-day date range (production shape)",
            f"{BASE}?search_query=cat:cs.AI+AND+submittedDate:%5B{YESTERDAY}0000+TO+{YESTERDAY}2359%5D&max_results=5")

### Interpretation

If shape 2 (`max_results=1`, no date range) succeeds just like shape 1, then a *small result count* is not the problem — plain smallness doesn't trigger a 406. If shapes 3 and 4 both fail (or both come back `x-cache: MISS, MISS`) regardless of whether the range is a full year or a single day, then it isn't "single-day ranges specifically" either — it's the presence of *any* `submittedDate:[...]` clause, because every such clause is uniquely parameterized and essentially never cached. That would reject this section's hypothesis, and confirm the cache-uniqueness explanation from Section 1 and `docs/406.md` as the real root cause — not the number of results requested, and not the width of the date window.

## 4. The actual fix: client-side date filtering

Given the above, the fix does not touch retries or request size at all. `ArxivClient.fetch_papers` now sends the plain, cacheable `cat:{category}` query (no `submittedDate:[...]` clause), over a wider window than requested, and filters to the requested date range **client-side** in Python. This cell calls the fixed, current `fetch_papers` with the exact same `from_date`/`to_date` arguments the Airflow `fetch_metadata` task uses — the call site is unchanged; only the internals of `ArxivClient.fetch_papers` changed.

In [ ]:
client = ArxivClient(ArxivSettings())
start = time.time()
papers = await client.fetch_papers(from_date=YESTERDAY, to_date=YESTERDAY, max_results=5)
elapsed = time.time() - start
print(f"OK in {elapsed:.1f}s - {len(papers)} papers for {YESTERDAY}")
for p in papers[:3]:
    print(f"  - {p.arxiv_id}: {p.title[:80]}")

## Management summary

- **Final state:** `fetch_papers(from_date=..., to_date=...)` now succeeds reliably (Section 4 above), because it never sends the uncacheable `submittedDate:[...]` query that caused the 406s.
- **Root causes:**
  1. Unescaped `[`/`]` in query URLs (fixed by percent-encoding).
  2. arXiv's export API answers a cache-miss with a bare 406 while throttling a host; a `submittedDate:[...]` clause is uniquely parameterized per call, so it (almost) never hits arXiv's shared cache and (almost) always hits the throttle check.
- **Rejected hypotheses:**
  - "arXiv changed its API rules" — the manual's allowed values were unchanged; the error message was itself a symptom of the overload, not a real validation failure.
  - "Only date-range queries are affected" — a bracket-free query with no date range also failed repeatedly under load, before Cause 2 was found.
  - "The retry budget is too short" (Section 2) — 0/6 live trials recovered even with a 20-attempt ceiling.
  - "Smaller requests are the problem" (Section 3) — a `max_results=1` request with no date range behaves like the broad control; the differentiator is the `submittedDate:[...]` clause, not result count or date-window width.

See [`docs/406.md`](../../docs/406.md) for the full write-up, and issues [#37](https://github.com/pkuppens/production-agentic-rag-course/issues/37), [#39](https://github.com/pkuppens/production-agentic-rag-course/issues/39), [#44](https://github.com/pkuppens/production-agentic-rag-course/issues/44), [#46](https://github.com/pkuppens/production-agentic-rag-course/issues/46).